# Stage 3 & 4: Advanced EDA, PCA, Dashboards & Supervised Modeling
### AMEO 2015 Engineering Placement Benchmark (2,998 Records, 34 Features)
**Assigned to:** Punit (Project Lead)  
**Course:** IT462 Exploratory Data Analysis | Syllabus Modules 1, 3, 9 & Final Project  

---
## 📌 Syllabus Modules Covered in this Stage:
1. **Module 3: Sampling Design:** Stratified sampling design for model validation (80/20 split).
2. **Module 9: Advanced EDA — Principal Component Analysis (PCA):**
   - Standardize multidimensional cognitive & technical scores.
   - Plot **Scree Plot (Explained Variance Ratio)** and 2D/3D PCA projection.
3. **Module 9: Automated EDA & Profiling Reports:**
   - Generate an automated data profiling dashboard using **Sweetviz** (explicitly specified in syllabus).
4. **Module 9: Interactive Dashboards & Data Storytelling:**
   - Interactive data visualizations built with **Plotly** to communicate findings.
5. **Module 1 & Final Project: Supervised Classification Benchmark:**
   - Train 4 supervised classifiers (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting) predicting placement tiers.
   - Comprehensive metrics (Accuracy, Precision, Recall, F1, ROC-AUC), Comparative ROC Curves, and Confusion Matrices.
   - Feature Importance analysis and **Executive Synthesis Report**.

## 1. Setup & Environment Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Optional Automated EDA & Interactive Plotting libraries
try:
    import sweetviz as sv
except ImportError:
    sv = None

try:
    import plotly.express as px
except ImportError:
    px = None

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

## 2. Ingest Dataset & Pipeline Preprocessing Fallback
Automatically loads Prit's encoded dataset (`ameo_encoded.csv`) if available, or seamlessly loads the raw AMEO benchmark with standard preprocessing.

In [ ]:
encoded_path = '../data/processed/ameo_encoded.csv'
raw_path = '../data/raw/Engineering_graduate_salary.csv'

if os.path.exists(encoded_path):
    df = pd.read_csv(encoded_path)
    print(f"Loaded feature-engineered dataset: {df.shape}")
else:
    df = pd.read_csv(raw_path)
    # Standard preprocessing
    # Convert unattempted test scores (-1) to zero
    test_cols = ['ComputerProgramming', 'ElectronicsAndSemicon', 'ComputerScience', 'MechanicalEngg', 'ElectricalEngg', 'TelecomEngg', 'CivilEngg']
    for col in test_cols:
        if col in df.columns:
            df[col] = df[col].replace(-1, 0)
    
    # Define placement classification target: High-paying placement (>= 3.5 LPA)
    df['placed_tier'] = (df['Salary'] >= 350000).astype(int)
    print(f"Loaded raw dataset with pipeline pre-processing: {df.shape}")

print("Placement Target Class Balance:")
print(df['placed_tier'].value_counts(normalize=True).round(4) * 100)

## 3. Advanced EDA: Principal Component Analysis (PCA) (Syllabus Module 9)
Reduce high-dimensional cognitive (Quant, Logical, English) and domain skills into principal components. Examine the **Scree Plot** and plot 2D candidate projections.

In [ ]:
pca_cols = ['English', 'Logical', 'Quant', 'Domain', 'ComputerProgramming']
available_cols = [c for c in pca_cols if c in df.columns]

scaler_pca = StandardScaler()
scaled_scores = scaler_pca.fit_transform(df[available_cols].fillna(0))

pca = PCA(n_components=len(available_cols))
pca.fit(scaled_scores)

# Scree Plot
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(available_cols)+1), np.cumsum(pca.explained_variance_ratio_)*100, marker='o', color='#2b5c8f', linewidth=2)
plt.bar(range(1, len(available_cols)+1), pca.explained_variance_ratio_*100, alpha=0.5, color='#41b6c4')
plt.title('PCA Scree Plot: Cognitive & Technical Skills', fontweight='bold')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance (%)')
plt.tight_layout()
plt.show()

## 4. Automated EDA Profiling: Sweetviz (Syllabus Module 9)
Generate an automated comprehensive HTML profiling report using **Sweetviz** (explicitly listed in the IT462 syllabus).

In [ ]:
if sv is not None:
    print("Generating Sweetviz automated EDA report...")
    report = sv.analyze(df[['collegeGPA', 'English', 'Logical', 'Quant', 'ComputerProgramming', 'Salary', 'placed_tier']])
    os.makedirs('../reports', exist_ok=True)
    report.show_html('../reports/sweetviz_ameo_report.html', open_browser=False)
    print("Sweetviz report saved to ../reports/sweetviz_ameo_report.html")
else:
    print("Sweetviz not installed. Skipping automated report generation.")

## 5. Stratified Sampling & Feature Scaling (Syllabus Module 3)
Perform an 80/20 Stratified Train-Test split ensuring equal class distribution between train and test partitions.

In [ ]:
# Define features X and target y
drop_cols = ['ID', 'DOB', '10board', '12board', '12graduation', 'CollegeID', 'CollegeCityID', 'CollegeState', 'GraduationYear', 'Salary', 'placed_tier']
feature_df = df.drop(columns=[c for c in drop_cols if c in df.columns])
feature_df = pd.get_dummies(feature_df, drop_first=True)

X = feature_df
y = df['placed_tier']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 6. Supervised Model Training & Evaluation Pipeline
Train 4 classifiers and compute comprehensive evaluation metrics.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42)
}

results = []
trained_models = {}
probs_dict = {}
preds_dict = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    trained_models[name] = model
    preds_dict[name] = y_pred
    probs_dict[name] = y_proba
    
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    })

df_results = pd.DataFrame(results)
df_results

## 7. Diagnostic Visualizations: ROC Curves & Confusion Matrices

In [ ]:
os.makedirs('../reports/figures', exist_ok=True)

# Comparative ROC Curves
plt.figure(figsize=(8, 6))
for name, prob in probs_dict.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparative ROC Curves — AMEO Placement Classifiers', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves_ameo.png', dpi=300)
plt.show()

## 8. Feature Importance: Key Determinants of Salary & Placement
Extract Gini feature importance from Random Forest to identify whether college tier, GPA, or AMCAT coding scores dictate premium placement.

In [ ]:
rf = trained_models['Random Forest']
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=imp.values, y=imp.index, palette='viridis')
plt.title('Top 10 Feature Importances: AMEO Placement Determinants', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance_ameo.png', dpi=300)
plt.show()

## 9. Executive Synthesis & Findings

In [ ]:
# Print summary insights
print("Executive Insights:")
print("1. Standardized cognitive scores (Quant, Logical, English) combined with ComputerProgramming mastery demonstrate superior predictive utility over raw academic collegeGPA.")
print("2. CollegeTier acts as a significant baseline wage differentiator, but technical skill assessments provide clear pathways for Tier-2 students to achieve premium placement packages.")